# Adaptive and Explainable Fraud Detection Under Concept Drift
IEEE-CIS Fraud Detection - clean version of the full project notebook.

**How to run it**
1. Add two inputs to the Kaggle notebook: the competition data and your `bundle-joblib` dataset (it holds the trained models, so you do not have to retrain).
2. Click **Run All**. With the saved models this takes roughly 15 to 25 minutes (loading data, drift analysis, scoring, SHAP).
3. Two switches in Part 0 control the slow parts:
   - `TRAIN_FROM_SCRATCH = True` retrains every model instead of loading them (about 1 hour).
   - `RUN_EXTRA_EXPERIMENTS = True` also runs the random-split model and the 5-seed noise test (about 6 extra minutes).

**Parts**
| Part | What it answers |
|---|---|
| 1 | Load the data and define the time periods |
| 2 | How much is missing, and does missing mean something? |
| 3 | Which columns do we keep? |
| 4 | First model, and random split versus time split |
| 5 | Four algorithms compared |
| 6 | What drifted? (PSI, missing-value shift, KS, Wasserstein) |
| 7 | Does retraining help? (fixed schedules, rolling window) |
| 8 | Can drift be detected early? Drift-triggered retraining |
| 9 | What if labels arrive late? |
| 10 | What does it cost? |
| 11 | Are the explanations (SHAP) stable? |
| 12 | Optional: how much does a result change with the random seed? |
| 13 | Final tables and figures |

## Part 0 - Setup

In [ ]:
import os, time, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
from scipy.stats import spearmanr, ks_2samp, wasserstein_distance
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
warnings.filterwarnings("ignore")

# ---- paths -------------------------------------------------------------
DATA_DIR   = "/kaggle/input/competitions/ieee-fraud-detection"
BUNDLE     = "/kaggle/input/datasets/kanishkbhuradiya/bundle-joblib/bundle.joblib"
REPORT_DIR = "/kaggle/working/report"

# ---- switches ----------------------------------------------------------
TRAIN_FROM_SCRATCH    = False   # True = retrain all models (about 1 hour). False = load saved models.
RUN_EXTRA_EXPERIMENTS = False   # True = also run the random-split model and the 5-seed test (about 6 min)

# ---- saved models and results -------------------------------------------
if TRAIN_FROM_SCRATCH:
    B = {}
    print("Training from scratch.")
else:
    B = joblib.load(BUNDLE)
    print("Loaded saved results:", len(B), "items,", len(B["model_cache"]), "models")

# ---- helper functions used in several places -----------------------------
def family(c):
    """Group a column into its feature family by name."""
    if c.startswith("V"): return "V"
    if c.startswith("C") and c[1:].isdigit(): return "C"
    if c.startswith("D") and c[1:].isdigit(): return "D"
    if c.startswith("M") and c[1:].isdigit(): return "M"
    if c.startswith("id_") or c in ("DeviceType", "DeviceInfo"): return "identity"
    if c.startswith("card"): return "card"
    return "other"

def psi(ref, cur, bins=10):
    """Population Stability Index of one feature: reference period vs current period (0 = no change)."""
    r = ref[~np.isnan(ref)]; c = cur[~np.isnan(cur)]
    if len(r) == 0 or len(c) == 0:
        return np.nan
    edges = np.unique(np.quantile(r, np.linspace(0, 1, bins + 1)))[1:-1]       # decile edges of the reference
    nb = len(edges) + 1
    rc = np.bincount(np.searchsorted(edges, r, side="right"), minlength=nb).astype(float)
    cc = np.bincount(np.searchsorted(edges, c, side="right"), minlength=nb).astype(float)
    rc = np.append(rc, np.isnan(ref).sum())                                    # extra bin for "missing"
    cc = np.append(cc, np.isnan(cur).sum())
    rp = np.clip(rc / rc.sum(), 1e-4, None)
    cp = np.clip(cc / cc.sum(), 1e-4, None)
    return float(np.sum((cp - rp) * np.log(cp / rp)))

## Part 1 - Load the data and define the time periods
All 590,540 transactions are used (we cannot sample, because time order matters). The data is sorted by time and cut into three periods:

| Period | Days | Weeks | Purpose |
|---|---|---|---|
| Training | 7 to 91 | 1 to 12 | the model learns here |
| Validation | 91 to 105 | 13 to 14 | early stopping, thresholds |
| Future test | 105 onward | 15 to 25 | the "future" we evaluate on |

The first 7 days are skipped as a warm-up. **Loading takes 1 to 3 minutes. Do not stop the cell.**

In [ ]:
t0 = time.time()
trans = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
ident = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
df = trans.merge(ident, on="TransactionID", how="left")      # keep every transaction, add identity info where it exists
del trans, ident
for c in df.select_dtypes("float64").columns:                # smaller number type = half the memory
    if c != "TransactionAmt":
        df[c] = df[c].astype("float32")

df = df.sort_values("TransactionDT").reset_index(drop=True)
df["day"]  = df["TransactionDT"] / 86400 - 1
df["week"] = (df["day"] // 7).astype(int)
print("Rows, columns:", df.shape, "|", round(time.time() - t0), "seconds | memory (GB):", round(df.memory_usage().sum() / 1e9, 2))

# short names used everywhere below
y    = df["isFraud"].values
week = df["week"].values
amt  = df["TransactionAmt"].values.astype("float64")
train_mask = (df["day"] >= 7)  & (df["day"] < 91)
val_mask   = (df["day"] >= 91) & (df["day"] < 105)
test_mask  =  df["day"] >= 105
test_weeks  = list(range(15, 26))
train_weeks = list(range(1, 13))
tw = np.isin(week, test_weeks)

def avg(scores_vec, weeks):
    """PR-AUC of a score vector (one value per transaction) over the given weeks."""
    m = np.isin(week, weeks)
    return average_precision_score(y[m], scores_vec[m])

weekly = df.groupby("week")["isFraud"].agg(transactions="size", frauds="sum")
weekly["fraud_rate_%"] = (weekly["frauds"] / weekly["transactions"] * 100).round(2)
print(weekly.T.to_string())

split_summary = pd.DataFrame({
    "rows":   [train_mask.sum(), val_mask.sum(), test_mask.sum()],
    "frauds": [df.loc[m, "isFraud"].sum() for m in (train_mask, val_mask, test_mask)]},
    index=["train", "validation", "test"])
split_summary["fraud_rate_%"] = (split_summary["frauds"] / split_summary["rows"] * 100).round(2)
print()
print(split_summary)

## Part 2 - Missing values
Many columns are mostly empty, but we do **not** delete those rows or fill the gaps blindly: in this data, "missing" is itself a clue. The cell below shows how much is missing per column family, and which columns' missing-ness best predicts fraud.

In [ ]:
# 1) how much is missing, by column family
miss = (df.drop(columns=["day", "week"]).isna().mean() * 100).round(1)
summary = pd.DataFrame({"missing_%": miss, "family": [family(c) for c in miss.index]})
summary["bucket"] = pd.cut(summary["missing_%"], bins=[-1, 0, 20, 50, 80, 95, 100],
                           labels=["0%", "0-20%", "20-50%", "50-80%", "80-95%", "95-100%"])
print("Number of columns per family and missing-% bucket")
print(pd.crosstab(summary["family"], summary["bucket"]))
print("\nAverage missing % per family")
print(summary.groupby("family")["missing_%"].mean().round(1))

# 2) fraud rate when the value is missing vs present
skip = {"isFraud", "TransactionID", "TransactionDT", "week", "day"}
rows = []
for c in df.columns:
    if c in skip:
        continue
    m = df[c].isna().values
    if m.mean() < 0.01 or m.mean() > 0.99:
        continue
    rows.append({"column": c, "missing_%": round(m.mean() * 100, 1),
                 "fraud_rate_when_missing_%": round(y[m].mean() * 100, 2),
                 "fraud_rate_when_present_%": round(y[~m].mean() * 100, 2)})
gap = pd.DataFrame(rows)
gap["difference"] = (gap["fraud_rate_when_missing_%"] - gap["fraud_rate_when_present_%"]).round(2)
gap = gap.reindex(gap["difference"].abs().sort_values(ascending=False).index)
print("\nOverall fraud rate: %.2f%%" % (y.mean() * 100))
print("Columns whose missing-ness predicts fraud most:")
print(gap.head(15).to_string(index=False))

## Part 3 - Choose the columns and build the model table `X`
Rules, all measured on the **training period only** (so the future cannot influence our choices):
1. Drop columns that are more than 95% empty, or constant.
2. Among the V columns, keep one only if it is not more than 95% correlated with a V column already kept (many V columns are near-copies).
3. Convert text columns to numbers, numbering each value in the order it **first appears in time**. Missing stays missing.

`TransactionID` and `TransactionDT` are never used as inputs.

In [ ]:
not_features = ["isFraud", "TransactionID", "TransactionDT", "week", "day"]
candidates = [c for c in df.columns if c not in not_features]
train = df.loc[train_mask, candidates]

# rule 1: nearly empty or constant
too_empty = train.isna().mean()[lambda s: s > 0.95].index.tolist()
constant  = train.nunique()[lambda s: s <= 1].index.tolist()
print("More than 95% missing:", len(too_empty), too_empty)
print("Constant:", len(constant), constant)

# rule 2: near-duplicate V columns
v_cols = [c for c in candidates if c.startswith("V") and c[1:].isdigit() and c not in too_empty and c not in constant]
sample = df.loc[train_mask, v_cols].sample(min(100_000, int(train_mask.sum())), random_state=42)
corr = sample.corr().abs()
kept = []
for c in v_cols:
    if all(corr.loc[c, k] <= 0.95 for k in kept):
        kept.append(c)
dropped_v = [c for c in v_cols if c not in kept]
print(f"V columns: {len(v_cols)} -> kept {len(kept)}, dropped {len(dropped_v)} as near-duplicates")

feature_cols = [c for c in candidates if c not in too_empty and c not in constant and c not in dropped_v]
text_cols = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]
print("Feature columns:", len(feature_cols), "| text columns:", len(text_cols))

# rule 3: text -> numbers (order of first appearance; missing stays missing)
X = df[feature_cols].copy()
for c in text_cols:
    codes, _ = pd.factorize(X[c])
    codes = codes.astype("float32")
    codes[codes < 0] = np.nan
    X[c] = codes
X = X.astype("float32")
print("X shape:", X.shape, "| all columns numeric:", bool((X.dtypes == "float32").all()))

if not TRAIN_FROM_SCRATCH:                                     # the saved models expect exactly these columns
    print("Same columns as the saved models:", feature_cols == B["feature_cols"])

## Part 4 - First model, and why a random split misleads
We use **LightGBM** and **PR-AUC** (how well real frauds are ranked above normal transactions). Accuracy is useless here: "never fraud" is already 96.5% accurate. A random model scores about 0.035 PR-AUC; a perfect one scores 1.0.

In [ ]:
X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

LGBM_PARAMS = dict(n_estimators=500, learning_rate=0.1, num_leaves=64, colsample_bytree=0.5,
                   subsample=0.8, subsample_freq=1, scale_pos_weight=5,
                   metric="average_precision", verbose=-1)        # used for all retraining experiments

if TRAIN_FROM_SCRATCH:
    model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=64, colsample_bytree=0.5,
                               subsample=0.8, subsample_freq=1, scale_pos_weight=5,
                               metric="average_precision", random_state=42, verbose=-1)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
    print("Trees used:", model.best_iteration_)
    val_score  = model.predict_proba(X_val)[:, 1]
    test_score = model.predict_proba(X_test)[:, 1]
else:
    val_score, test_score = B["val_score"], B["test_score"]

print("PR-AUC validation:", round(average_precision_score(y_val, val_score), 3))
print("PR-AUC future test (all weeks):", round(average_precision_score(y_test, test_score), 3))

test_df = pd.DataFrame({"week": week[test_mask], "y": y_test, "score": test_score})
print("\nPR-AUC per future week")
print(test_df.groupby("week").apply(lambda g: pd.Series({"transactions": len(g), "frauds": int(g.y.sum()),
      "PR_AUC": round(average_precision_score(g.y, g.score), 3)})).to_string())

In [ ]:
# ---- random split (same model, same settings) vs the honest time split ----
if RUN_EXTRA_EXPERIMENTS or TRAIN_FROM_SCRATCH:
    idx = np.arange(len(y))
    tr_i, rest_i = train_test_split(idx, test_size=0.30, random_state=42, stratify=y)
    va_i, te_i   = train_test_split(rest_i, test_size=2/3, random_state=42, stratify=y[rest_i])   # 70% / 10% / 20%, mixed over time
    model_r = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=64, colsample_bytree=0.5,
                                 subsample=0.8, subsample_freq=1, scale_pos_weight=5,
                                 metric="average_precision", random_state=42, verbose=-1)
    model_r.fit(X.iloc[tr_i], y[tr_i], eval_set=[(X.iloc[va_i], y[va_i])], callbacks=[lgb.early_stopping(50, verbose=False)])
    pr_random = average_precision_score(y[te_i], model_r.predict_proba(X.iloc[te_i])[:, 1])
else:
    pr_random = 0.816     # value from our earlier run (set RUN_EXTRA_EXPERIMENTS = True to recompute it)
    print("Using the earlier value for the random split (not recomputed).")

pr_future = average_precision_score(y_test, test_score)
print("PR-AUC with RANDOM split :", round(pr_random, 3))
print("PR-AUC with TIME split   :", round(pr_future, 3))
print("The random split looks", round((pr_random / pr_future - 1) * 100), "% better")

# ---- is the early-vs-late drop real? (bootstrap) ----
early = test_df[test_df.week.isin([15, 16, 17])]
late  = test_df[test_df.week.isin([23, 24, 25])]
rng = np.random.RandomState(0)
diffs = []
for _ in range(200):
    e = early.sample(len(early), replace=True, random_state=rng.randint(1e9))
    l = late.sample(len(late),  replace=True, random_state=rng.randint(1e9))
    diffs.append(average_precision_score(e.y, e.score) - average_precision_score(l.y, l.score))
print("\nEarly (weeks 15-17) PR-AUC:", round(average_precision_score(early.y, early.score), 3))
print("Late  (weeks 23-25) PR-AUC:", round(average_precision_score(late.y, late.score), 3))
print("Drop = %.3f, 95%% range: %.3f to %.3f" % (np.mean(diffs), np.quantile(diffs, .025), np.quantile(diffs, .975)))

## Part 5 - Four algorithms compared
Logistic Regression, Random Forest, XGBoost and LightGBM, all trained once on the training period and then frozen. The drop column (early weeks minus late weeks) shows how much each loses over time.

In [ ]:
if TRAIN_FROM_SCRATCH:
    scores = {"lightgbm": test_score}

    m_xgb = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.5,
                              min_child_weight=3, scale_pos_weight=5, tree_method="hist", eval_metric="aucpr",
                              early_stopping_rounds=50, n_jobs=-1, random_state=42)
    m_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    scores["xgboost"] = m_xgb.predict_proba(X_test)[:, 1]

    m_rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, max_features="sqrt", max_samples=0.5,
                                  class_weight="balanced_subsample", n_jobs=-1, random_state=42)
    m_rf.fit(X_train.fillna(-999).values, y_train)                         # forests need no empty cells
    scores["random_forest"] = m_rf.predict_proba(X_test.fillna(-999).values)[:, 1]

    numeric  = [c for c in X.columns if c not in text_cols]
    low_card = [c for c in text_cols if df[c].nunique() <= 30]             # small categories -> one-hot; huge ones are left out
    prep = ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", QuantileTransformer(n_quantiles=200, subsample=100_000,
                                                        output_distribution="normal", random_state=42))]), numeric),
        ("cat", Pipeline([("fill", SimpleImputer(strategy="constant", fill_value=-1)),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), low_card)])
    m_lr = Pipeline([("prep", prep), ("clf", LogisticRegression(C=0.1, max_iter=300, class_weight="balanced"))])
    m_lr.fit(X_train, y_train)
    scores["logreg"] = m_lr.predict_proba(X_test)[:, 1]
else:
    scores = B["scores"]

wk_t = week[test_mask]
early_t, late_t = np.isin(wk_t, [15, 16, 17]), np.isin(wk_t, [23, 24, 25])
base = pd.DataFrame({n: {"all weeks": average_precision_score(y_test, s),
                         "weeks 15-17": average_precision_score(y_test[early_t], s[early_t]),
                         "weeks 23-25": average_precision_score(y_test[late_t], s[late_t])} for n, s in scores.items()}).T
base["drop"] = base["weeks 15-17"] - base["weeks 23-25"]
base["relative drop %"] = (base["drop"] / base["weeks 15-17"] * 100).round(0)
print("PR-AUC of the four frozen models")
print(base.round(3).sort_values("all weeks", ascending=False).to_string())

print("\nPR-AUC per future week")
print(pd.DataFrame({n: {w: average_precision_score(y_test[wk_t == w], s[wk_t == w]) for w in test_weeks}
                    for n, s in scores.items()}).round(3).to_string())

## Part 6 - What drifted?
**PSI** (Population Stability Index) compares a feature's distribution in the training period with each future week. Rule of thumb: above 0.1 is a moderate change, above 0.25 a large one. PSI can be fooled by narrow or unstable features, so each feature also gets a **noise floor**: how much its PSI varies between training weeks, where nothing should have changed. Only the part above that floor counts as drift (`excess`).

In [ ]:
Xtr, wk_tr = X[train_mask].values, week[train_mask]
Xte, wk_te = X[test_mask].values,  week[test_mask]

rows = []
for j, col in enumerate(X.columns):
    a = Xtr[:, j]
    drift = [psi(a, Xte[wk_te == w, j]) for w in test_weeks]                          # each future week vs the training period
    noise = [psi(a[wk_tr != w], a[wk_tr == w]) for w in train_weeks]                   # each training week vs the rest of training
    rows.append({"feature": col, "psi_future": np.nanmean(drift), "noise_floor": np.nanquantile(noise, 0.95)})
res = pd.DataFrame(rows)
res["excess"] = (res["psi_future"] - res["noise_floor"]).clip(lower=0)
res["family"] = res["feature"].map(family)
res = res.sort_values("excess", ascending=False).reset_index(drop=True)
del Xtr, Xte

print("Top 15 drifting features (PSI above their own noise floor):")
print(res.head(15).round(3).to_string(index=False))
print("\nFeatures with PSI > 0.1  and above noise:", int(((res.psi_future > 0.1) & (res.excess > 0)).sum()), "of", len(res))
print("Features with PSI > 0.25 and above noise:", int(((res.psi_future > 0.25) & (res.excess > 0)).sum()))
print("\nAverage excess PSI by family:")
print(res.groupby("family")["excess"].mean().round(3).sort_values(ascending=False))

**What changed in those features?** The drifting groups are columns that are filled in or left empty together. The next cell checks whether it is the *availability* (missing share) or the *values* that changed, and whether the link to fraud changed (fraud rate when missing vs present), using `V4` as the example group.

In [ ]:
# 1) share of missing values per week (V4 and D11 lie exactly on top of each other; addr1 is the stable control)
check = ["M1", "M7", "V4", "D11", "addr1"]
weekly_missing = pd.DataFrame({c: df[c].isna().groupby(df["week"]).mean() * 100 for c in check}).round(1)
print(weekly_missing.T.to_string())
ax = weekly_missing.plot(figsize=(10, 4.5), marker="o")
ax.axvspan(0.5, 12.5, color="green", alpha=0.07); ax.axvspan(12.5, 14.5, color="orange", alpha=0.10); ax.axvspan(14.5, 25.5, color="red", alpha=0.05)
ax.set_xlabel("week (green = train, orange = validation, red = future test)"); ax.set_ylabel("% missing")
ax.set_title("Share of missing values per week"); plt.show()

# 2) fraud rate when missing vs present, by period
periods = {"1 train early (wk1-6)": (1, 6), "2 train late (wk7-12)": (7, 12), "3 validation (wk13-14)": (13, 14),
           "4 test early (wk15-19)": (15, 19), "5 test late (wk20-25)": (20, 25)}
rows = []
for c in ["M1", "M7", "V4", "D11", "addr1"]:
    is_miss = df[c].isna()
    for name, (a, b) in periods.items():
        m = df["week"].between(a, b)
        rows.append({"column": c, "period": name, "missing_%": round(is_miss[m].mean() * 100, 1),
                     "fraud_%_if_missing": round(df.loc[m & is_miss, "isFraud"].mean() * 100, 2),
                     "fraud_%_if_present": round(df.loc[m & ~is_miss, "isFraud"].mean() * 100, 2)})
print()
print(pd.DataFrame(rows).to_string(index=False))

# 3) where does the first model lose accuracy? (V4 missing vs present, early vs late test weeks)
test_df["V4_missing"] = df.loc[test_mask, "V4"].isna().values
test_df["period"] = np.where(test_df.week <= 19, "early (wk15-19)", "late (wk20-25)")
rows = []
for (flag, per), g in test_df.groupby(["V4_missing", "period"]):
    pr = average_precision_score(g.y, g.score)
    rows.append({"V4": "missing" if flag else "present", "period": per, "rows": len(g), "frauds": int(g.y.sum()),
                 "fraud_rate_%": round(g.y.mean() * 100, 2), "PR_AUC": round(pr, 3), "lift_vs_random": round(pr / g.y.mean(), 1)})
print("\nModel quality by V4 group (lift = PR-AUC divided by the fraud rate)")
print(pd.DataFrame(rows).to_string(index=False))

**Cross-check with two more measures.** PSI includes a "missing" bin. **KS** (largest gap between two distributions) and **Wasserstein** (how far values must move) compare only the values that exist, so they show whether the *values* changed. Each also gets a noise floor.

In [ ]:
rng = np.random.RandomState(0)
num_feats = [c for c in feature_cols if c not in text_cols]
tr_idx, te_idx = np.where(train_mask)[0], np.where(test_mask)[0]
tr_wk, te_wk = week[tr_idx], week[te_idx]
Xtr = X.iloc[tr_idx][num_feats].values
Xte = X.iloc[te_idx][num_feats].values

def ks_wd(ref, cur, scale):
    """KS statistic and (scaled) Wasserstein distance between two samples, ignoring missing values."""
    r, c = ref[~np.isnan(ref)], cur[~np.isnan(cur)]
    if len(r) < 50 or len(c) < 50 or not scale > 0:
        return np.nan, np.nan
    if len(r) > 50_000: r = rng.choice(r, 50_000, replace=False)
    if len(c) > 50_000: c = rng.choice(c, 50_000, replace=False)
    return ks_2samp(r, c).statistic, wasserstein_distance(r, c) / scale

rows = []
for j, f in enumerate(num_feats):
    a = Xtr[:, j]
    nn = a[~np.isnan(a)]
    if len(nn) < 50:
        continue
    scale = np.subtract(*np.quantile(nn, [0.75, 0.25]))                 # typical spread of the feature
    if scale <= 0:
        scale = nn.std()
    drift = [ks_wd(a, Xte[te_wk == w, j], scale) for w in test_weeks]
    noise = [ks_wd(a[tr_wk != w], a[tr_wk == w], scale) for w in train_weeks]
    rows.append({"feature": f, "ks": np.nanmean([d[0] for d in drift]), "wd": np.nanmean([d[1] for d in drift]),
                 "ks_noise": np.nanquantile([n[0] for n in noise], 0.95), "wd_noise": np.nanquantile([n[1] for n in noise], 0.95)})
ks_table = pd.DataFrame(rows).set_index("feature")
ks_table["ks_excess"] = (ks_table.ks - ks_table.ks_noise).clip(lower=0)
ks_table["wd_excess"] = (ks_table.wd - ks_table.wd_noise).clip(lower=0)
del Xtr, Xte

drift_cmp = res.set_index("feature")[["family", "psi_future", "excess"]].rename(columns={"excess": "psi_excess"}).join(ks_table, how="inner")
drift_cmp["missing_shift_pp"] = (X.loc[test_mask, drift_cmp.index].isna().mean() - X.loc[train_mask, drift_cmp.index].isna().mean()) * 100

show = ["family", "psi_excess", "ks", "ks_excess", "wd_excess", "missing_shift_pp"]
for title, col in [("PSI excess (includes the missing bin)", "psi_excess"), ("KS excess (values only)", "ks_excess"), ("Wasserstein excess", "wd_excess")]:
    print(f"Top 12 by {title}:")
    print(drift_cmp.sort_values(col, ascending=False)[show].head(12).round(3).to_string(), "\n")

print("Agreement between the three rankings (Spearman over", len(drift_cmp), "features):")
print(drift_cmp[["psi_excess", "ks_excess", "wd_excess"]].corr(method="spearman").round(2).to_string())
top20 = lambda col: set(drift_cmp[col].nlargest(20).index)
print("Top-20 overlap  PSI-KS:", len(top20("psi_excess") & top20("ks_excess")), "| PSI-Wasserstein:", len(top20("psi_excess") & top20("wd_excess")),
      "| KS-Wasserstein:", len(top20("ks_excess") & top20("wd_excess")))
flag = (drift_cmp.psi_future > 0.25) & (drift_cmp.psi_excess > 0)
print("\nThe", int(flag.sum()), "PSI-flagged features vs the others (median):")
print(drift_cmp.groupby(flag)[["ks_excess", "wd_excess", "missing_shift_pp"]].median().round(3).rename(index={True: "PSI-flagged", False: "others"}).to_string())

## Part 7 - Does retraining help?
A walk-forward simulation: at the start of each future week a strategy decides whether to retrain **using only data from before that week**, then scores that week.

| Strategy | Meaning |
|---|---|
| static | train once at week 15, never update |
| every 4 weeks / every 2 weeks | retrain on all data so far |
| rolling 8 weeks, every 2 | retrain on only the latest 8 weeks of data |

Each retraining uses the two most recent weeks for early stopping. Models already stored in the bundle are reused, so no training happens unless `TRAIN_FROM_SCRATCH = True`.

In [ ]:
model_cache = {} if TRAIN_FROM_SCRATCH else dict(B["model_cache"])      # (week, window) -> trained model

def train_for(r, window=None):
    """The model retrained at week r, using only data from BEFORE week r (trains it if not stored yet)."""
    key = (r, window)
    if key not in model_cache:
        lo = 1 if window is None else max(1, r - 2 - window)
        tr, va = (week >= lo) & (week <= r - 3), np.isin(week, [r - 2, r - 1])
        t0 = time.time()
        m = lgb.LGBMClassifier(**LGBM_PARAMS, random_state=42)
        m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], callbacks=[lgb.early_stopping(30, verbose=False)])
        print(f"  trained for week {r} | window={window} | train weeks {lo}-{r-3} | rows={tr.sum():,} | {round(time.time()-t0)}s")
        model_cache[key] = m
    return model_cache[key]

def run_strategy(every, window=None):
    """Fraud scores for all future rows when retraining every `every` weeks."""
    out = np.full(len(df), np.nan)
    for w in test_weeks:
        r = 15 + ((w - 15) // every) * every                    # most recent retraining point
        idx = np.where(week == w)[0]
        out[idx] = train_for(r, window).predict_proba(X.iloc[idx])[:, 1]
    return out

strategies = {"static":              dict(every=99, window=None),
              "every 4 weeks":       dict(every=4,  window=None),
              "every 2 weeks":       dict(every=2,  window=None),
              "rolling 8w, every 2": dict(every=2,  window=8)}
strategy_scores = {name: run_strategy(**kw) for name, kw in strategies.items()}
n_retrains = {name: len({15 + ((w - 15) // kw["every"]) * kw["every"] for w in test_weeks}) for name, kw in strategies.items()}

def recall_at_budget(s, weeks, frac):
    """Share of all frauds caught if only the top `frac` highest-scored transactions of each week are reviewed."""
    caught = total = 0
    for w in weeks:
        idx = np.where(week == w)[0]
        top = np.argsort(-s[idx])[:int(round(frac * len(idx)))]
        caught += y[idx][top].sum(); total += y[idx].sum()
    return caught / total

def summarize(names):
    t = pd.DataFrame({
        "retrains":             {n: n_retrains[n] for n in names},
        "PR-AUC all weeks":     {n: avg(strategy_scores[n], test_weeks) for n in names},
        "PR-AUC weeks 15-17":   {n: avg(strategy_scores[n], [15, 16, 17]) for n in names},
        "PR-AUC weeks 23-25":   {n: avg(strategy_scores[n], [23, 24, 25]) for n in names},
        "recall top 5% (all)":  {n: recall_at_budget(strategy_scores[n], test_weeks, 0.05) for n in names},
        "recall top 5% (late)": {n: recall_at_budget(strategy_scores[n], [23, 24, 25], 0.05) for n in names}})
    t["gain vs static %"] = ((t["PR-AUC all weeks"] / t.loc["static", "PR-AUC all weeks"] - 1) * 100).round(1)
    return t

per_week = pd.DataFrame({n: {w: avg(s, [w]) for w in test_weeks} for n, s in strategy_scores.items()})
print("PR-AUC per future week"); print(per_week.round(3).to_string())
print("\nSummary"); print(summarize(list(strategies)).round(3).to_string())

In [ ]:
# are the gains over the static model more than luck? (bootstrap on future transactions)
rng = np.random.RandomState(0)
rows_test = np.where(tw)[0]
late_rows = np.where(np.isin(week, [23, 24, 25]))[0]

def boot_diff(rows, name, other="static", n=100):
    diffs = []
    for _ in range(n):
        i = rng.choice(rows, len(rows), replace=True)
        if y[i].sum() == 0:
            continue
        diffs.append(average_precision_score(y[i], strategy_scores[name][i]) - average_precision_score(y[i], strategy_scores[other][i]))
    return np.mean(diffs), np.quantile(diffs, 0.025), np.quantile(diffs, 0.975)

out = []
for name in ["every 4 weeks", "every 2 weeks", "rolling 8w, every 2"]:
    m_all, lo_all, hi_all = boot_diff(rows_test, name)
    m_late, lo_late, hi_late = boot_diff(late_rows, name)
    out.append({"strategy": name, "weeks better than static": f"{(per_week[name] > per_week['static']).sum()}/{len(test_weeks)}",
                "gain, all weeks": round(m_all, 3), "95% range": f"{lo_all:.3f} to {hi_all:.3f}",
                "gain, weeks 23-25": round(m_late, 3), "95% range (late)": f"{lo_late:.3f} to {hi_late:.3f}"})
print(pd.DataFrame(out).to_string(index=False))
m4, lo4, hi4 = boot_diff(rows_test, "every 2 weeks", other="every 4 weeks")
print("\nevery 2 weeks minus every 4 weeks: %.3f  (95%% range %.3f to %.3f)" % (m4, lo4, hi4))

**Precision, recall and F1 at a fixed alert threshold**, chosen on each model's own validation weeks (the threshold that maximises F1). Also: how much fraud is caught if investigators can review only the top 3% or top 5% of transactions each week.

In [ ]:
def best_f1_threshold(yv, sv):
    p, r, t = precision_recall_curve(yv, sv)
    f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
    return t[int(np.argmax(f1))]

thr_cache = {}
def threshold_for(r, window):
    if (r, window) not in thr_cache:
        va = np.isin(week, [r - 2, r - 1])
        thr_cache[(r, window)] = best_f1_threshold(y[va], model_cache[(r, window)].predict_proba(X[va])[:, 1])
    return thr_cache[(r, window)]

def threshold_metrics(name, kw, weeks):
    tp = fp = fn = n = 0
    for w in weeks:
        r = 15 + ((w - 15) // kw["every"]) * kw["every"]
        idx = np.where(week == w)[0]
        flag, yy = strategy_scores[name][idx] >= threshold_for(r, kw["window"]), y[idx]
        tp += int((flag & (yy == 1)).sum()); fp += int((flag & (yy == 0)).sum()); fn += int((~flag & (yy == 1)).sum()); n += len(idx)
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"precision": round(prec, 3), "recall": round(rec, 3), "F1": round(2 * prec * rec / max(prec + rec, 1e-12), 3),
            "alerts per 1000 txns": round(1000 * (tp + fp) / n, 1)}

for label, weeks in [("ALL FUTURE WEEKS (15-25)", test_weeks), ("LATE WEEKS (23-25)", [23, 24, 25])]:
    print(label)
    print(pd.DataFrame({n: threshold_metrics(n, kw, weeks) for n, kw in strategies.items()}).T.to_string(), "\n")

print("Share of fraud caught when only the top X% of each week's transactions can be reviewed")
print(pd.DataFrame({n: {"all weeks, top 3%": recall_at_budget(strategy_scores[n], test_weeks, 0.03),
                        "all weeks, top 5%": recall_at_budget(strategy_scores[n], test_weeks, 0.05),
                        "late weeks, top 3%": recall_at_budget(strategy_scores[n], [23, 24, 25], 0.03),
                        "late weeks, top 5%": recall_at_budget(strategy_scores[n], [23, 24, 25], 0.05)} for n in strategies}).T.round(3).to_string())

## Part 8 - Can drift be detected early?
Two warning signals that **need no labels**, tested on the static model against its real weekly accuracy:
- **Score PSI**: has the distribution of the model's scores moved?
- **Feature PSI**: has the distribution of its 20 most important features moved (above their noise floor)?

If a signal worked, it would rise in the weeks where PR-AUC is low.

In [ ]:
m_static = model_cache[(15, None)]
val_rows = np.isin(week, [13, 14])
val_scores = m_static.predict_proba(X[val_rows])[:, 1]                 # the model's scores in its own validation weeks
val_pr = average_precision_score(y[val_rows], val_scores)
top20 = list(X.columns[np.argsort(-m_static.booster_.feature_importance(importance_type="gain"))[:20]])
noise_floor = dict(zip(res["feature"], res["noise_floor"]))
ref = X.loc[val_rows]
print("Top 5 features of the static model (by gain):", top20[:5])
print("Static model PR-AUC on its own validation weeks:", round(val_pr, 3), "\n")

rows = []
for w in test_weeks:
    idx = np.where(week == w)[0]
    s = strategy_scores["static"][idx]
    excess = [max(psi(ref[f].values, X.iloc[idx][f].values) - noise_floor[f], 0.0) for f in top20]
    pr = average_precision_score(y[idx], s)
    rows.append({"week": w, "score_PSI": round(psi(val_scores, s), 3), "mean_score": round(float(s.mean()), 3),
                 "feature_PSI_above_noise": round(float(np.mean(excess)), 3), "PR_AUC_static": round(pr, 3),
                 "relative_PR_AUC_drop": round(1 - pr / val_pr, 3)})
signals = pd.DataFrame(rows)
print("Mean score in validation weeks:", round(float(val_scores.mean()), 3))
print(signals.to_string(index=False))
for col in ["score_PSI", "feature_PSI_above_noise"]:
    r = spearmanr(signals[col], signals.PR_AUC_static)
    print(f"Spearman({col}, PR-AUC) = {r.correlation:.2f}  (p={r.pvalue:.3f}, n={len(signals)} weeks)")

**Drift-triggered retraining.** Since label-free signals did not work, the trigger uses labels: each week, look at how the *current* model did on the *previous* week. If its PR-AUC fell by more than a threshold (relative to its own validation PR-AUC), retrain now. A 2-week cooldown prevents back-to-back retrains. All three thresholds are reported, not just the best one.

In [ ]:
val_pr_cache = {}
def val_pr_of(r):
    """PR-AUC of the model retrained at week r on its own validation weeks."""
    if r not in val_pr_cache:
        va = np.isin(week, [r - 2, r - 1])
        val_pr_cache[r] = average_precision_score(y[va], model_cache[(r, None)].predict_proba(X[va])[:, 1])
    return val_pr_cache[r]

def run_triggered(thr, cooldown=2, lag=1):
    out = np.full(len(df), np.nan)
    r, retrain_weeks = 15, [15]
    for w in test_weeks:
        if w > 15 and (w - r) >= cooldown and (w - lag) >= r:
            idx = np.where(week == w - lag)[0]                               # the most recent week we can judge
            if 1 - average_precision_score(y[idx], out[idx]) / val_pr_of(r) > thr:
                r = w
                retrain_weeks.append(w)
        idx = np.where(week == w)[0]
        out[idx] = train_for(r, None).predict_proba(X.iloc[idx])[:, 1]
    return out, retrain_weeks

trigger_weeks = {}
for thr in [0.10, 0.15, 0.25]:
    name = f"trigger >{int(thr*100)}% drop"
    strategy_scores[name], trigger_weeks[name] = run_triggered(thr)
    n_retrains[name] = len(trigger_weeks[name])
    print(f"{name}: retrained at weeks {trigger_weeks[name]}")

summary2 = summarize(list(strategy_scores))
print()
print(summary2.round(3).to_string())

## Part 9 - What if labels arrive late?
Real fraud labels (chargebacks) take weeks. Here a transaction's label is known only **4 weeks** later, so a model retrained at week `r` can only learn from data up to week `r - 1 - 4`.

In [ ]:
D = 4                                                                  # label delay in weeks
delay_cache = {} if TRAIN_FROM_SCRATCH else dict(B["delay_cache"])

def train_for_d(r):
    """The model retrained at week r when only labels up to week r-1-D are known."""
    key = (r, D)
    if key not in delay_cache:
        tr, va = (week >= 1) & (week <= r - 3 - D), np.isin(week, [r - 2 - D, r - 1 - D])
        m = lgb.LGBMClassifier(**LGBM_PARAMS, random_state=42)
        m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], callbacks=[lgb.early_stopping(30, verbose=False)])
        print(f"  trained for week {r} | train weeks 1-{r-3-D} | rows={tr.sum():,}")
        delay_cache[key] = m
    return delay_cache[key]

def run_delayed(every):
    out = np.full(len(df), np.nan)
    for w in test_weeks:
        idx = np.where(week == w)[0]
        out[idx] = train_for_d(15 + ((w - 15) // every) * every).predict_proba(X.iloc[idx])[:, 1]
    return out

delayed = {name: run_delayed(every) for name, every in [("static", 99), ("every 4 weeks", 4), ("every 2 weeks", 2)]}

tab = pd.DataFrame({n: {"PR-AUC, no delay": avg(strategy_scores[n], test_weeks), f"PR-AUC, {D}w delay": avg(delayed[n], test_weeks),
                        "late PR-AUC, no delay": avg(strategy_scores[n], [23, 24, 25]), f"late PR-AUC, {D}w delay": avg(delayed[n], [23, 24, 25]),
                        f"recall top 5% late, {D}w delay": recall_at_budget(delayed[n], [23, 24, 25], 0.05)} for n in delayed}).T
print(tab.round(3).to_string())
print("\nGain over the static model (PR-AUC, all weeks):")
for n in ["every 4 weeks", "every 2 weeks"]:
    print(f"  {n}: no delay {tab.loc[n, 'PR-AUC, no delay'] - tab.loc['static', 'PR-AUC, no delay']:+.3f} | "
          f"{D}w delay {tab.loc[n, f'PR-AUC, {D}w delay'] - tab.loc['static', f'PR-AUC, {D}w delay']:+.3f}")

## Part 10 - What does it cost?
The dataset has **no real cost information**, so these four scenarios are *assumptions* (amounts in dollars):

| Scenario | False alarm costs | Missed fraud costs |
|---|---|---|
| cheap review | $2 | 100% of the amount |
| balanced | $10 | 100% of the amount + $15 |
| friction-heavy | $25 + 2% of the amount | 100% of the amount + $15 |
| fraud-heavy | $5 | 200% of the amount + $25 |

For each model and scenario, the alert threshold is chosen to minimise cost on that model's **validation** weeks, then the cost is measured on the future weeks. "Break-even" = how much one extra retrain may cost before it stops paying off.

In [ ]:
scenarios = {"cheap review":   dict(fp_fixed=2,  fp_pct=0.00, fn_pct=1.0, fn_fixed=0),
             "balanced":       dict(fp_fixed=10, fp_pct=0.00, fn_pct=1.0, fn_fixed=15),
             "friction-heavy": dict(fp_fixed=25, fp_pct=0.02, fn_pct=1.0, fn_fixed=15),
             "fraud-heavy":    dict(fp_fixed=5,  fp_pct=0.00, fn_pct=2.0, fn_fixed=25)}

def cost_vectors(yy, aa, sc):
    fp = (1 - yy) * (sc["fp_fixed"] + sc["fp_pct"] * aa)               # cost if a legitimate transaction is flagged
    fn = yy * (sc["fn_pct"] * aa + sc["fn_fixed"])                     # cost if a fraud is missed
    return fp, fn

def best_cost_threshold(yv, sv, av, sc):
    """The score threshold with the lowest total cost on a labelled (validation) set."""
    o = np.argsort(-sv, kind="mergesort")
    fp, fn = cost_vectors(yv[o], av[o], sc)
    total_fn = fn.sum()
    cost = np.r_[total_fn, np.cumsum(fp) + (total_fn - np.cumsum(fn))]   # cost of flagging the top m transactions
    m = int(np.argmin(cost))
    return np.inf if m == 0 else sv[o][m - 1]

def model_key(name, w):
    """Which stored model produced the scores of strategy `name` in week w."""
    if name == "static":              return (15, None)
    if name == "every 4 weeks":       return (15 + ((w - 15) // 4) * 4, None)
    if name == "every 2 weeks":       return (15 + ((w - 15) // 2) * 2, None)
    if name == "rolling 8w, every 2": return (15 + ((w - 15) // 2) * 2, 8)
    return (max(x for x in trigger_weeks[name] if x <= w), None)        # drift-triggered strategies

thr_cost = {}
def cost_threshold(key, scname):
    if (key, scname) not in thr_cost:
        va = np.isin(week, [key[0] - 2, key[0] - 1])
        sv = model_cache[key].predict_proba(X[va])[:, 1]
        thr_cost[(key, scname)] = best_cost_threshold(y[va], sv, amt[va], scenarios[scname])
    return thr_cost[(key, scname)]

cost_names = list(strategy_scores)
total_cost, no_model = {n: {} for n in cost_names}, {}
for scname, sc in scenarios.items():
    no_model[scname] = cost_vectors(y[tw], amt[tw], sc)[1].sum()         # cost if nothing is ever flagged
    for n in cost_names:
        c = 0.0
        for w in test_weeks:
            idx = np.where(week == w)[0]
            flag = strategy_scores[n][idx] >= cost_threshold(model_key(n, w), scname)
            fp, fn = cost_vectors(y[idx], amt[idx], sc)
            c += fp[flag].sum() + fn[~flag].sum()
        total_cost[n][scname] = c
costs = pd.DataFrame(total_cost).T
costs.loc["(no model)"] = pd.Series(no_model)

print("Total cost over the future weeks ($)"); print(costs.round(0).astype(int).to_string())
sav = (1 - costs.loc[cost_names].div(costs.loc["static"], axis=1)) * 100
print("\nSaving vs the STATIC model (%)"); print(sav.round(1).to_string())
print("\nSaving of the static model vs having NO model (%)"); print(((1 - costs.loc["static"] / costs.loc["(no model)"]) * 100).round(1).to_string())

print("\nBreak-even cost per extra retrain, 'balanced' scenario:")
for n in cost_names[1:]:
    extra, saved = n_retrains[n] - 1, costs.loc["static", "balanced"] - costs.loc[n, "balanced"]
    if extra > 0:
        print(f"  {n:22s} saves ${saved:>9,.0f} with {extra} extra retrains -> break-even ${saved / extra:>8,.0f} per retrain")

## Part 11 - Are the explanations stable? (SHAP)
SHAP shows how much each feature pushed each prediction toward or away from fraud. The average absolute value tells how much the model *relies* on a feature. We ask:
- **A.** Does the static model rely on the same features in the training weeks, early test weeks and late test weeks? (Compared with a *noise baseline*: two samples from the same period.)
- **B.** Does the retrained model rely on different features than the static one on the same late rows, and is that more than a different random seed would cause?

The SHAP values come from LightGBM's built-in exact calculation (`pred_contrib=True`).

In [ ]:
rng = np.random.RandomState(0)
m_static = model_cache[(15, None)]       # trained once on weeks 1-12
m_late   = model_cache[(25, None)]       # retrained on weeks 1-22

def sample_rows(a, b, n=5000, fraud_only=False):
    mask = (week >= a) & (week <= b)
    if fraud_only:
        mask &= (y == 1)
    idx = np.where(mask)[0]
    return rng.choice(idx, min(n, len(idx)), replace=False)

def mean_abs_shap(model, idx):
    sv = model.booster_.predict(X.iloc[idx], pred_contrib=True)[:, :-1]      # last column is the baseline, not a feature
    return pd.Series(np.abs(sv).mean(axis=0), index=X.columns)

def stability(a, b, k=20):
    """(Spearman correlation over all features, overlap of the top-k features)."""
    ta, tb = set(a.nlargest(k).index), set(b.nlargest(k).index)
    return round(spearmanr(a, b).correlation, 3), round(len(ta & tb) / len(ta | tb), 3)

periods_s = {"train (wk 1-12)": (1, 12), "test early (wk 15-17)": (15, 17), "test late (wk 23-25)": (23, 25)}
imp_all   = pd.DataFrame({n: mean_abs_shap(m_static, sample_rows(a, b)) for n, (a, b) in periods_s.items()})
imp_fraud = pd.DataFrame({n: mean_abs_shap(m_static, sample_rows(a, b, 2000, True)) for n, (a, b) in periods_s.items()})

share = imp_all / imp_all.sum() * 100
top = imp_all["train (wk 1-12)"].nlargest(12).index
print("A1. Share of total importance (%), static model, all transactions")
print(share.loc[top].round(1).to_string())

base_b = mean_abs_shap(m_static, sample_rows(1, 12))                         # a second sample of the SAME period = noise baseline
rows = [{"comparison": "train vs train (noise baseline)", "all rows": stability(imp_all["train (wk 1-12)"], base_b), "fraud rows": "-"}]
for n in ["test early (wk 15-17)", "test late (wk 23-25)"]:
    rows.append({"comparison": f"train vs {n}", "all rows": stability(imp_all["train (wk 1-12)"], imp_all[n]),
                 "fraud rows": stability(imp_fraud["train (wk 1-12)"], imp_fraud[n])})
print("\nA2. Stability as (Spearman over all features, top-20 overlap)")
print(pd.DataFrame(rows).to_string(index=False))

print("\nA3. Share of importance by feature family (%)")
print(share.groupby(share.index.map(family)).sum().round(1).to_string())

late_idx = sample_rows(23, 25)
s_static, s_late = mean_abs_shap(m_static, late_idx), mean_abs_shap(m_late, late_idx)
shift = (s_late / s_late.sum() - s_static / s_static.sum()) * 100
print("\nB. Static vs retrained model on the same late rows (Spearman, top-20 overlap):", stability(s_static, s_late))
print("Relies on MORE (change in importance share, % points):"); print(shift.nlargest(6).round(2).to_string())
print("Relies on LESS:"); print(shift.nsmallest(6).round(2).to_string())

In [ ]:
# ---- Check A: direction of the effect (does a higher value still push toward fraud?) ----
top_feats = list(imp_all["train (wk 1-12)"].nlargest(10).index)
col_index = {c: j for j, c in enumerate(X.columns)}
rows = []
for n, (a, b) in periods_s.items():
    idx = sample_rows(a, b, 4000)
    sv = m_static.booster_.predict(X.iloc[idx], pred_contrib=True)[:, :-1]
    for f in top_feats:
        xv = X[f].values[idx]; ok = ~np.isnan(xv)
        rows.append({"feature": f, "period": n, "direction": spearmanr(xv[ok], sv[ok, col_index[f]]).correlation})
dirs = pd.DataFrame(rows).pivot(index="feature", columns="period", values="direction")[list(periods_s)]
print("Spearman(feature value, SHAP value): + means a higher value -> more fraud-like")
print("(for text-coded columns the sign has no direct meaning; only stability does)")
print(dirs.loc[top_feats].round(2).to_string())

# ---- Check B: how much does the explanation change from a different random seed alone? ----
tr15, va15 = (week >= 1) & (week <= 12), np.isin(week, [13, 14])
m_seed = lgb.LGBMClassifier(**LGBM_PARAMS, random_state=7)
m_seed.fit(X[tr15], y[tr15], eval_set=[(X[va15], y[va15])], callbacks=[lgb.early_stopping(30, verbose=False)])
s_seed = mean_abs_shap(m_seed, late_idx)
shift_seed = (s_seed / s_seed.sum() - s_static / s_static.sum()) * 100
print("\nStatic vs SAME-DATA model with a different seed:", stability(s_static, s_seed))
print("Static vs RETRAINED model:                      ", stability(s_static, s_late))
print("\nLargest changes caused by the seed alone (% points):");  print(shift_seed.reindex(shift_seed.abs().nlargest(6).index).round(2).to_string())
print("Largest changes caused by retraining (% points):");        print(shift.reindex(shift.abs().nlargest(6).index).round(2).to_string())

## Part 12 - Optional: run-to-run noise
Trains the static model five times with different random seeds on exactly the same data. The spread of the results is the size of difference that should **not** be interpreted. Runs only if `RUN_EXTRA_EXPERIMENTS = True` (about 4 to 5 minutes).

In [ ]:
seeds = None
if RUN_EXTRA_EXPERIMENTS or TRAIN_FROM_SCRATCH:
    tr_s, va_s = (week >= 1) & (week <= 12), np.isin(week, [13, 14])
    y_tw, wk_tw = y[tw], week[tw]
    e, l = np.isin(wk_tw, [15, 16, 17]), np.isin(wk_tw, [23, 24, 25])
    rows = []
    for seed in [1, 2, 3, 4, 5]:
        m = lgb.LGBMClassifier(**LGBM_PARAMS, random_state=seed)
        m.fit(X[tr_s], y[tr_s], eval_set=[(X[va_s], y[va_s])], callbacks=[lgb.early_stopping(30, verbose=False)])
        s = m.predict_proba(X[tw])[:, 1]
        rows.append({"seed": seed, "trees": m.best_iteration_,
                     "PR-AUC all weeks": average_precision_score(y_tw, s),
                     "PR-AUC early": average_precision_score(y_tw[e], s[e]),
                     "PR-AUC late": average_precision_score(y_tw[l], s[l])})
    seeds = pd.DataFrame(rows)
    seeds["early minus late"] = seeds["PR-AUC early"] - seeds["PR-AUC late"]
    print(seeds.round(3).to_string(index=False))
    print("\nmean / std / min / max")
    print(seeds.drop(columns="seed").agg(["mean", "std", "min", "max"]).round(3).to_string())
else:
    print("Skipped (set RUN_EXTRA_EXPERIMENTS = True to run it).")

## Part 13 - Final tables and figures
Saves five figures and the main tables to `/kaggle/working/report/` (they also appear below).

In [ ]:
os.makedirs(REPORT_DIR, exist_ok=True)

def save_show(name):
    plt.savefig(f"{REPORT_DIR}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

# Figure 1: random split vs time split
fig, ax = plt.subplots(figsize=(4.8, 3.8))
ax.bar(["random split", "time split\n(future)"], [pr_random, pr_future], color=["#9aa5b1", "#d1495b"])
for i, v in enumerate([pr_random, pr_future]):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center")
ax.set_ylabel("PR-AUC"); ax.set_title("Same model, two evaluation protocols")
save_show("fig1_random_vs_time_split")

# Figure 2: weekly PR-AUC of the four frozen models
wk_tbl = pd.DataFrame({n: [average_precision_score(y_test[wk_t == w], s[wk_t == w]) for w in test_weeks] for n, s in scores.items()}, index=test_weeks)
ax = wk_tbl.plot(marker="o", figsize=(8, 4)); ax.set_xlabel("future week"); ax.set_ylabel("PR-AUC"); ax.set_title("Frozen models lose accuracy over time")
save_show("fig2_baseline_models_by_week")

# Figure 3: weekly PR-AUC by retraining strategy
ax = per_week.plot(marker="o", figsize=(8, 4)); ax.set_xlabel("future week"); ax.set_ylabel("PR-AUC"); ax.set_title("Retraining recovers accuracy")
save_show("fig3_retraining_strategies_by_week")

# Figure 4: share of missing values over time
ax = weekly_missing.plot(marker="o", figsize=(9, 4))
ax.axvspan(0.5, 12.5, color="green", alpha=0.07); ax.axvspan(12.5, 14.5, color="orange", alpha=0.10); ax.axvspan(14.5, 25.5, color="red", alpha=0.05)
ax.set_xlabel("week (green = train, orange = validation, red = future test)"); ax.set_ylabel("% missing"); ax.set_title("Several feature groups became much more complete")
save_show("fig4_missing_share_over_time")

# Figure 5: cost saving vs static
ax = sav.drop("static").plot.barh(figsize=(9, 5)); ax.axvline(0, color="k")
ax.set_xlabel("% cost saved vs the static model (assumed cost scenarios)"); ax.set_title("Retraining reduces estimated cost")
save_show("fig5_cost_saving_vs_static")

# tables as CSV files
tables = {"table_baseline_models": base, "table_retraining_strategies": summary2, "table_cost_saving_vs_static": sav,
          "table_drift_measures": drift_cmp, "table_seed_variation": seeds}
for name, t in tables.items():
    if t is not None:
        t.to_csv(f"{REPORT_DIR}/{name}.csv")
print("Saved in", REPORT_DIR, ":", sorted(os.listdir(REPORT_DIR)))

## Optional - save the trained models for next time
Only needed after a `TRAIN_FROM_SCRATCH = True` run. Download `bundle.joblib` from the Output panel and upload it as a Kaggle Dataset, then point `BUNDLE` in Part 0 to it.

In [ ]:
if TRAIN_FROM_SCRATCH:
    os.makedirs("/kaggle/working/frozen", exist_ok=True)
    joblib.dump({"model_cache": model_cache, "delay_cache": delay_cache, "scores": scores, "test_score": test_score,
                 "val_score": val_score, "feature_cols": feature_cols, "text_cols": text_cols},
                "/kaggle/working/frozen/bundle.joblib", compress=3)
    print("saved:", round(os.path.getsize("/kaggle/working/frozen/bundle.joblib") / 1e6), "MB")
else:
    print("Nothing to save (models were loaded from the bundle).")